import pandas as pd
import os

### Q1: Build Your Personalized Knowledge Base

**Rules applied for Roll Number 1024160029:**
- Last two digits are `2` and `9`.
- **Digit 2**: category `["billing", "account", "general"][2 % 3]` -> `general`. Question: "what is your customer support contact information", Answer: "You can email support at contact@college.edu or call +1-800-555-0199.", Keywords: "contact support helpline".
- **Digit 9**: category `["billing", "account", "general"][9 % 3]` -> `billing`. Question: "how can i request a refund for an overpayment", Answer: "Please submit a refund request form along with your payment receipt to the billing office.", Keywords: "refund receipt overpayment".

In [6]:
#Ques 1
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

personalized_entries = [
    # Digit 2: Category "general"
    {
        "question": "what is your customer support contact information",
        "answer": "You can email support at contact@college.edu or call +1-800-555-0199.",
        "keywords": "contact support helpline",
        "category": "general"
    },
    # Digit 9: Category "billing"
    {
        "question": "how can i request a refund for an overpayment",
        "answer": "Please submit a refund request form along with your payment receipt to the billing office.",
        "keywords": "refund receipt overpayment",
        "category": "billing"
    }
]

# Combine into a single DataFrame
df = pd.DataFrame(fixed_entries + personalized_entries)
print("Personalized FAQ DataFrame:")
df

Personalized FAQ DataFrame:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,what is your customer support contact information,You can email support at contact@college.edu o...,contact support helpline,general
5,how can i request a refund for an overpayment,Please submit a refund request form along with...,refund receipt overpayment,billing


In [7]:
#Ques 2
def score_query(query, df):
    query_words = set(query.lower().strip().split())
    results = []
    for idx, row in df.iterrows():
        keyword_words = set(row['keywords'].lower().split())
        score = len(query_words.intersection(keyword_words))
        if score > 0:
            results.append({
                "index": idx,
                "question": row['question'],
                "answer": row['answer'],
                "category": row['category'],
                "score": score
            })
    res_df = pd.DataFrame(results)
    if not res_df.empty:
        res_df = res_df.sort_values(by='score', ascending=False).reset_index(drop=True)
    return res_df

print("Matching entries for query 'how to pay':")
score_query("how to pay", df)

Matching entries for query 'how to pay':


,index,question,answer,category,score
0,3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,1


In [8]:
#Ques 3
def same_category(category_name, df):
    filtered_df = df[df['category'].str.lower() == category_name.lower()]
    return filtered_df['question']

test_category = df.loc[4, 'category'] # index 4 is category "general"
print(f"Questions in category '{test_category}':")
same_category(test_category, df)

Questions in category 'general':


2                          what are your working hours
4    what is your customer support contact information
Name: question, dtype: object

In [9]:
#Ques 4
def add_keyword_and_save(df, entry_index, new_keyword, roll_number="1024160029"):
    current_keywords = df.loc[entry_index, 'keywords']
    keywords_list = current_keywords.split()
    if new_keyword.strip().lower() not in [k.lower() for k in keywords_list]:
        df.loc[entry_index, 'keywords'] = f"{current_keywords} {new_keyword.strip()}"
    
    filename = f"{roll_number}_faq_data.csv"
    df.to_csv(filename, index=False)
    print(f"Keyword '{new_keyword}' added to entry {entry_index}.")
    print(f"DataFrame saved to '{filename}' successfully!")

# Ask user for a keyword (we will prompt or run with a mock value for script compatibility)
new_keyword = input("Enter a new keyword: ") if 'IPython' in globals() else "refund_policy"
add_keyword_and_save(df, 5, new_keyword)

Keyword 'refund_policy' added to entry 5.
DataFrame saved to '1024160029_faq_data.csv' successfully!


In [10]:
#Ques 5
print("FAQ Entries count per category:")
df.groupby('category').size()

FAQ Entries count per category:


category
account    1
billing    3
general    2
dtype: int64

In [11]:
#Ques 6
def score_query_v2(query, df):
    query_words = set(query.lower().strip().split())
    results = []
    for idx, row in df.iterrows():
        keyword_words = set(row['keywords'].lower().split())
        score = len(query_words.intersection(keyword_words))
        if score > 0:
            results.append({
                "index": idx,
                "question": row['question'],
                "answer": row['answer'],
                "category": row['category'],
                "score": score
            })
    res_df = pd.DataFrame(results)
    if res_df.empty:
        print(f"Query: '{query}' -> No matches found.")
        return res_df
        
    res_df = res_df.sort_values(by='score', ascending=False).reset_index(drop=True)
    max_score = res_df['score'].max()
    highest_matches = res_df[res_df['score'] == max_score]
    
    if len(highest_matches) > 1:
        print(f"Query: '{query}' -> [TIE DETECTED] Multiple entries matched with highest score {max_score}:")
        for _, row in highest_matches.iterrows():
            print(f"  - Question: '{row['question']}' (Category: {row['category']})")
    else:
        print(f"Query: '{query}' -> [SINGLE BEST MATCH] Found with score {max_score}:")
        row = highest_matches.iloc[0]
        print(f"  - Question: '{row['question']}' (Category: {row['category']})")
    return res_df

print("--- Test 1: Query causing a tie (matches 'fee' keyword in multiple entries) ---")
score_query_v2("fee", df)

print("\n--- Test 2: Query with single match (no tie) ---")
score_query_v2("password login", df);

--- Test 1: Query causing a tie (matches 'fee' keyword in multiple entries) ---
Query: 'fee' -> [TIE DETECTED] Multiple entries matched with highest score 1:
  - Question: 'what is the annual fee' (Category: billing)
  - Question: 'how can i pay the fee' (Category: billing)

--- Test 2: Query with single match (no tie) ---
Query: 'password login' -> [SINGLE BEST MATCH] Found with score 2:
  - Question: 'how to reset password' (Category: account)
